<a href="https://colab.research.google.com/github/Debbie1236-cmd/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Debbie1236-cmd/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This section scores every page in the test set using the Week 5/6 validated model, ranks them from most to least likely to be declining, and sorts the top slice into named archetypes with a suggested action — the same way a triage nurse turns a severity score into "see a doctor now" vs. "can wait." Reason-code thresholds (what counts as "stale" or "low CTR") are defined using training data only, never the test set, to avoid quietly leaking test information into the rules.

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("https://raw.githubusercontent.com/Debbie1236-cmd/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

feature_cols = ["days_since_last_update", "avg_position", "has_position_data",
                 "search_volume", "competition", "has_keyword_data", "ctr", "engagement_rate"]

def build_features(d):
    d = d.copy()
    d["is_declining_label"] = (d["trend_direction"] == "down").astype(int)
    d["has_keyword_data"] = d["search_volume"].notna().astype(int)
    d["has_position_data"] = (d["avg_position"] != 0).astype(int)
    d["search_volume"] = d["search_volume"].fillna(0)
    d["competition"] = d["competition"].fillna(0)
    return d

# same client-grouped split as Week 6
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_df, test_df = build_features(df.iloc[train_idx]), build_features(df.iloc[test_idx])

# same model as Week 6
Xtr_raw, y_train = train_df[feature_cols], train_df["is_declining_label"]
scaler = StandardScaler().fit(Xtr_raw)
model = LogisticRegression(max_iter=1000, random_state=42).fit(scaler.transform(Xtr_raw), y_train)
test_df["decline_probability"] = model.predict_proba(scaler.transform(test_df[feature_cols]))[:, 1]

# rank the queue and split into priority tiers (top 10% high, next 20% medium, rest low)
queue = test_df.sort_values("decline_probability", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
n = len(queue)
queue["priority_tier"] = np.where(queue["rank"] <= n * 0.10, "high",
                            np.where(queue["rank"] <= n * 0.30, "medium", "low"))

# thresholds defined on TRAINING data only
ctr_median_visible_train = train_df.loc[train_df["has_position_data"] == 1, "ctr"].median()
stale_threshold_train = train_df["days_since_last_update"].quantile(0.75)

def reason_codes(row):
    codes = []
    if row["has_position_data"] == 0:
        codes.append("no_visibility_data")
        return codes
    if row["ctr"] < ctr_median_visible_train:
        codes.append("low_ctr_visible_page")
    if row["days_since_last_update"] >= stale_threshold_train:
        codes.append("stale_content")
    if row["search_volume"] > 0:
        codes.append("has_search_demand")
    if not codes:
        codes.append("no_flags_visible_page")
    return codes

queue["reason_codes"] = queue.apply(reason_codes, axis=1)
queue["reason_codes_str"] = queue["reason_codes"].apply(lambda c: "|".join(c))

def archetype_and_action(row):
    codes, tier = row["reason_codes"], row["priority_tier"]
    if "no_visibility_data" in codes:
        return "untracked_no_visibility_data", "audit_for_baseline_data"
    if tier == "high" and "low_ctr_visible_page" in codes and "has_search_demand" in codes:
        return "declining_visible_underclicked", "refresh_and_review_ctr"
    if tier == "high" and "stale_content" in codes:
        return "declining_and_stale", "refresh"
    if tier == "high":
        return "declining_flagged_by_model", "refresh_review"
    if tier == "medium":
        return "watch_list", "monitor"
    return "low_risk", "no_action_needed"

queue[["archetype", "suggested_action"]] = queue.apply(
    lambda r: pd.Series(archetype_and_action(r)), axis=1
)

print(queue.groupby(["archetype", "suggested_action"]).size().sort_values(ascending=False))
print()
print(queue[["rank", "decline_probability", "priority_tier", "archetype", "suggested_action", "reason_codes_str"]].head(8))


archetype                       suggested_action       
low_risk                        no_action_needed           4253
watch_list                      monitor                    1232
declining_visible_underclicked  refresh_and_review_ctr      295
declining_and_stale             refresh                     262
untracked_no_visibility_data    audit_for_baseline_data      62
declining_flagged_by_model      refresh_review               59
dtype: int64

   rank  decline_probability priority_tier                       archetype  \
0     1             0.691185          high             declining_and_stale   
1     2             0.687722          high             declining_and_stale   
2     3             0.687161          high  declining_visible_underclicked   
3     4             0.686504          high             declining_and_stale   
4     5             0.686383          high  declining_visible_underclicked   
5     6             0.685441          high  declining_visible_underclicked   


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who this is for: an internal FlyRank content reviewer deciding which pages to look at first this cycle not a public-facing tool, not a client deliverable on its own.

What it does: ranks pages by predicted decline risk and groups the top ones into named archetypes with a suggested next action, so a human can triage faster.

Validated performance: precision@50 measured 0.68 under a client-grouped split (Week 6) of the top 50 flagged pages, about 68% actually matched the declining pattern on clients the model had never seen. That's the honest number this playbook is built on.

Where it stops being valid: this was trained and tested on one anonymized snapshot 30,000 pages, 32 clients, trailing-90-day metrics. It hasn't been tested on a different time period, a different client base, or pages outside this dataset. About 4% of pages have no tracking data at all and can't be meaningfully scored  they need a manual baseline check, not a refresh recommendation.

In [22]:
no_visibility_rate = (df["avg_position"] == 0).mean()
class_balance = train_df["is_declining_label"].mean()
print(f"Clients in dataset: {df['client_id'].nunique()}")
print(f"Rows with no visibility/position data: {no_visibility_rate:.1%}")
print(f"Declining-label rate (training data): {class_balance:.1%}")
print(f"Validated precision@50 (Week 6, client-grouped split): 0.68")


Clients in dataset: 32
Rows with no visibility/position data: 4.0%
Declining-label rate (training data): 55.0%
Validated precision@50 (Week 6, client-grouped split): 0.68


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A human must check, before acting on any recommendation:

Read the actual page does the recommendation still make sense in context?
Check if the page is tied to an active campaign, a legal/compliance requirement, or a scheduled change already in motion
Confirm the client relationship is active and in scope

Never automate no-go list:

Never auto-publish or auto-edit content based on this queue
Never auto-delete or unpublish a page based on this queue
Never use this score alone to evaluate a writer's or editor's performance
Never apply this model's score to a client outside the 32 in this dataset without re-checking it first
Never treat "no tracking data" pages as low priority by default  they need a manual audit, not silence

In [23]:
no_go_list = [
    "auto-publish or auto-edit content",
    "auto-delete or unpublish pages",
    "evaluate individual writer/editor performance",
    "score a client outside the training population without revalidation",
    "silently deprioritize 'no_visibility_data' pages",
]
for item in no_go_list:
    print("NEVER:", item)


NEVER: auto-publish or auto-edit content
NEVER: auto-delete or unpublish pages
NEVER: evaluate individual writer/editor performance
NEVER: score a client outside the training population without revalidation
NEVER: silently deprioritize 'no_visibility_data' pages


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

What would tell us this playbook has gone stale:

The share of pages with no tracking data drifts far from the current ~4% baseline
The declining-label rate drifts far from the current ~54% baseline (a sign the content population itself has changed)
When re-checked against real outcomes later, precision@50 drops meaningfully below the validated 0.68

When to retrain: on a fixed quarterly schedule regardless, or immediately if any of the above shifts happen, or if a new client is added that the model hasn't seen before.

In [24]:
monitoring_baseline = {
    "no_visibility_rate": round(no_visibility_rate, 4),
    "declining_label_rate": round(class_balance, 4),
    "validated_precision_at_50": 0.68,
    "ctr_median_visible_train": round(ctr_median_visible_train, 4),
    "stale_threshold_days_train": float(stale_threshold_train),
}
print(monitoring_baseline)

{'no_visibility_rate': np.float64(0.0402), 'declining_label_rate': np.float64(0.5501), 'validated_precision_at_50': 0.68, 'ctr_median_visible_train': 0.09, 'stale_threshold_days_train': 104.0}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked queue is saved to work/outputs/ (regenerated each run, not committed the repo's leak-guard blocks data files). The key numbers are saved as a small JSON file, and one summary chart is saved to work/figures/ both of these ARE committed, since they're what the Week 8 paper will point back to.

In [25]:
import os, json
import matplotlib.pyplot as plt

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

queue.to_csv("../outputs/w07_ranked_queue.csv", index=False)

metrics = {
    "validated_precision_at_50": 0.68,
    "test_set_size": len(queue),
    "priority_tier_counts": queue["priority_tier"].value_counts().to_dict(),
    "archetype_counts": queue["archetype"].value_counts().to_dict(),
    "monitoring_baseline": monitoring_baseline,
}
with open("../outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

action_counts = queue["suggested_action"].value_counts()
plt.figure(figsize=(6, 4))
action_counts.plot(kind="barh")
plt.xlabel("Number of pages")
plt.title("Suggested actions across the ranked queue")
plt.tight_layout()
plt.savefig("../figures/w07_action_mix.svg")
plt.close()

print("Exported: w07_ranked_queue.csv, w07_metrics.json, w07_action_mix.svg")


Exported: w07_ranked_queue.csv, w07_metrics.json, w07_action_mix.svg


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.